# CS781 Phase 6 — Experiment 4: Model 1 + Houlsby Adapters (Lean)

**Goal:** a compact, time-capped experiment intended to finish within ~90 minutes on a Kaggle T4. Model 1 is treated as a lower-priority architecture, so this notebook avoids exhaustive tuning.

- Reuses the Phase 1 Model 1 checkpoint and tokenizer.
- Houlsby adapters after attention and FFN sublayers; base weights frozen.
- One epoch on a capped 900-document training subset.
- Shorter 512-token source + 256-token target training sequences.
- 6-document / 2-configuration decoding tune; 16-document untouched final validation.
- All 100 official test documents are generated.
- Selects decoding by **BERTScore F1**, matching the competition's primary metric.

**Runtime is a target, not a guarantee.** Actual time depends heavily on Model 1's cached generation speed on the T4. The test set must contain all 100 records.

In [1]:
import os, re, json, math, time, random, zipfile, shutil, glob
from pathlib import Path
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

SEED = 2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if DEVICE.type=="cuda" else "CPU")
assert DEVICE.type == "cuda", "Use a Kaggle GPU session (T4 recommended)."

Device: cuda | GPU: Tesla T4


In [3]:
# ---- Paths: adjust only if your Kaggle dataset mount differs.
ROOT = Path("/kaggle/working/phase6_exp4_model1_houlsby")
ROOT.mkdir(parents=True, exist_ok=True)
CKPT_DIR = Path("/kaggle/input/phase1model-wts")  # update if Kaggle mount has a different name
TOKENIZER_CANDIDATES = [
    Path("/kaggle/input/datasets/aaryan1234567890/phase1model-wts/tokenizer.json"),
    Path("/kaggle/input/datasets/aaryan1234567890/phase1-complete-model/tokenizer.json"),
    Path("/kaggle/input/phase1model-wts/tokenizer.json"),
    Path("/kaggle/input/phase1-complete-model/tokenizer.json"),
    Path("/kaggle/working/cs781_encoder_120m/data/tokenizer.json"),
]
BEST_CKPT_CANDIDATES = [
    Path("/kaggle/input/datasets/aaryan1234567890/phase1model-wts/enc_full_best (3).pt"),
    Path("/kaggle/input/datasets/aaryan1234567890/phase1model-wts/enc_full_best.pt"),
    Path("/kaggle/input/datasets/aaryan1234567890/phase1-complete-model/enc_full_best (9).pt"),
    Path("/kaggle/input/phase1model-wts/enc_full_best (3).pt"),
    Path("/kaggle/input/phase1model-wts/enc_full_best.pt"),
    Path("/kaggle/input/phase1-complete-model/enc_full_best (9).pt"),
]
TOKENIZER_PATH = next((p for p in TOKENIZER_CANDIDATES if p.exists()), None)
BASE_CKPT = next((p for p in BEST_CKPT_CANDIDATES if p.exists()), None)
# Fallback discovery for Kaggle's dataset-specific mount layout.
if TOKENIZER_PATH is None:
    hits=list(Path("/kaggle/input").rglob("tokenizer (1).json"))
    TOKENIZER_PATH=hits[0] if hits else None
if BASE_CKPT is None:
    hits=list(Path("/kaggle/input").rglob("enc_full_best*.pt"))
    BASE_CKPT=hits[0] if hits else None
print("Tokenizer:", TOKENIZER_PATH)
print("Base checkpoint:", BASE_CKPT)
if TOKENIZER_PATH is None:
    found = list(Path("/kaggle/input").rglob("tokenizer.json"))
    print("Tokenizer candidates found:", found[:10])
if BASE_CKPT is None:
    found = list(Path("/kaggle/input").rglob("*enc*best*.pt"))
    print("Checkpoint candidates found:", found[:10])
assert TOKENIZER_PATH is not None, "Attach the Phase 1 Model 1 tokenizer as a Kaggle dataset."
assert BASE_CKPT is not None, "Attach the Phase 1 Model 1 best checkpoint as a Kaggle dataset."

# Private Phase 6 dataset. Add HF_TOKEN in Kaggle Secrets if repository access requires it.
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

from huggingface_hub import hf_hub_download
TRAIN_JSON = hf_hub_download(HF_DATASET_REPO, "Phase_6_Summ/Train/train.json",
                             repo_type="dataset", token=HF_TOKEN)
TEST_JSON = hf_hub_download(HF_DATASET_REPO, "Phase_6_Summ/Test/test.json",
                            repo_type="dataset", token=HF_TOKEN)
print("Train:", TRAIN_JSON, "| Test:", TEST_JSON)

Tokenizer: /kaggle/input/datasets/aaryant23/model1-ckpts/tokenizer (1).json
Base checkpoint: /kaggle/input/datasets/aaryant23/model1-ckpts/enc_full_best (9).pt


Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

Train: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_6_Summ/Train/train.json | Test: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_6_Summ/Test/test.json


In [4]:
# ---- Lightweight tokenizer loader: same LegalBPE implementation from Phase 1.
BPE_SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]
class LegalBPE:
    def __init__(self, obj):
        self.vocab_size = obj["vocab_size"]
        self.merges = [list(x) for x in obj["merges"]]
        self.itos = list(obj["itos"])
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.symbol_bytes = {f"{b:02x}":bytes([b]) for b in range(256)}
        for l,r in self.merges: self.symbol_bytes[l+r] = self.symbol_bytes[l]+self.symbol_bytes[r]
        self.ranks = {(l,r):i for i,(l,r) in enumerate(self.merges)}
    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)
    def encode_word(self, word):
        syms=[f"{b:02x}" for b in word.encode("utf-8")]
        while len(syms)>1:
            best=None; rank=None
            for j in range(len(syms)-1):
                r=self.ranks.get((syms[j],syms[j+1]))
                if r is not None and (rank is None or r<rank):
                    best=(syms[j],syms[j+1]); rank=r
            if best is None: break
            l,r=best; out=[]; j=0
            while j<len(syms):
                if j+1<len(syms) and syms[j]==l and syms[j+1]==r:
                    out.append(l+r); j+=2
                else: out.append(syms[j]); j+=1
            syms=out
        return syms
    def encode(self, text):
        out=[]
        for unit in re.findall(r"\S+|\s+", text):
            out.extend(self.stoi.get(s,self.unk_token_id) for s in self.encode_word(unit))
        return out
    def decode(self, ids):
        buf=bytearray()
        for idx in ids:
            tok=self.itos[int(idx)]
            if tok not in BPE_SPECIALS: buf.extend(self.symbol_bytes[tok])
        return buf.decode("utf-8", errors="replace")

tokenizer = LegalBPE(json.load(open(TOKENIZER_PATH, encoding="utf-8")))
print("Tokenizer vocab:", len(tokenizer), "roundtrip:", tokenizer.decode(tokenizer.encode("Legal test ₹100"))=="Legal test ₹100")
assert len(tokenizer) == 24576, f"Unexpected vocab size: {len(tokenizer)}"

Tokenizer vocab: 24576 roundtrip: True


In [5]:
# ---- Load and inspect Phase 6 data.
train_raw = json.load(open(TRAIN_JSON, encoding="utf-8"))
test_raw = json.load(open(TEST_JSON, encoding="utf-8"))
def normalize_records(obj, is_test=False):
    if isinstance(obj, dict):
        rows=[]
        for k,v in obj.items():
            if isinstance(v,dict):
                rows.append({"id":str(v.get("id",k)), "document":v.get("document",v.get("text","")),
                             "summary":v.get("summary","")})
        return rows
    rows=[]
    for i,v in enumerate(obj):
        rows.append({"id":str(v.get("id",i)), "document":v.get("document",v.get("text","")),
                     "summary":v.get("summary","")})
    return rows
train_rows=[r for r in normalize_records(train_raw) if r["document"].strip() and r["summary"].strip()]
test_rows=normalize_records(test_raw, True)
print("Usable train:",len(train_rows),"test:",len(test_rows))
assert len(test_rows)==100, f"Expected 100 test rows, found {len(test_rows)}"
# Duplicate-grouped deterministic split; reserve disjoint tuning and final-validation rows.
rng=random.Random(SEED)
groups={}
for r in train_rows:
    key=re.sub(r"\s+"," ",r["document"]).strip()[:1200].lower()
    groups.setdefault(key,[]).append(r)
group_list=list(groups.values()); rng.shuffle(group_list)
val_groups=[]; tr_groups=[]
for g in group_list:
    if sum(len(x) for x in val_groups)<160: val_groups.append(g)
    else: tr_groups.append(g)
val_all=[r for g in val_groups for r in g]
train_all=[r for g in tr_groups for r in g]
rng.shuffle(val_all); rng.shuffle(train_all)
decode_rows=val_all[:6]
final_val_rows=val_all[6:22]
# Random subset for runtime and representativeness; avoids over-selecting very long documents.
train_all=rng.sample(train_all, min(900, len(train_all)))
print("Train subset:",len(train_all),"decode tune:",len(decode_rows),"final val:",len(final_val_rows))
assert len(final_val_rows)>=16 and len(decode_rows)==6

Usable train: 7028 test: 100
Train subset: 900 decode tune: 6 final val: 16


In [7]:
# ---- Model 1 architecture from Phase 1 + Houlsby adapters.
D_MODEL=768; N_LAYERS=14; N_HEADS=12; FFN_DIM=3072; MAX_POSITIONS=2048
DROPOUT=0.0
ADAPTER_DIM=64
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self,d_model,max_positions):
        super().__init__()
        pos=torch.arange(max_positions,dtype=torch.float32).unsqueeze(1)
        div=torch.exp(torch.arange(0,d_model,2,dtype=torch.float32)*(-math.log(10000.0)/d_model))
        pe=torch.zeros(max_positions,d_model)
        pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer("pe",pe,persistent=False)
    def forward(self,position_ids): return self.pe[position_ids]

class HoulsbyAdapter(nn.Module):
    def __init__(self, hidden, bottleneck=64):
        super().__init__()
        self.down=nn.Linear(hidden,bottleneck)
        self.act=nn.GELU()
        self.up=nn.Linear(bottleneck,hidden)
        nn.init.normal_(self.down.weight,std=0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self,x): return x+self.up(self.act(self.down(x)))

class PrefixSelfAttention(nn.Module):
    def __init__(self,d_model,n_heads,dropout):
        super().__init__(); self.n_heads=n_heads; self.head_dim=d_model//n_heads
        self.qkv=nn.Linear(d_model,3*d_model); self.out_proj=nn.Linear(d_model,d_model); self.dropout=dropout
    def forward(self,x,attn_mask=None,past_kv=None,use_cache=False):
        B,T,C=x.shape
        q,k,v=self.qkv(x).view(B,T,3,self.n_heads,self.head_dim).unbind(dim=2)
        q=q.transpose(1,2); k=k.transpose(1,2); v=v.transpose(1,2)
        if past_kv is not None:
            pk,pv=past_kv; k=torch.cat([pk,k],dim=2); v=torch.cat([pv,v],dim=2)
        y=F.scaled_dot_product_attention(q,k,v,attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,is_causal=False)
        y=y.transpose(1,2).contiguous().view(B,T,C)
        return self.out_proj(y), ((k,v) if use_cache else None)

class EncoderBlock(nn.Module):
    def __init__(self,d_model,n_heads,ffn_dim,dropout):
        super().__init__()
        self.ln1=nn.LayerNorm(d_model); self.attn=PrefixSelfAttention(d_model,n_heads,dropout)
        self.ln2=nn.LayerNorm(d_model)
        self.ff=nn.Sequential(nn.Linear(d_model,ffn_dim),nn.GELU(),
                              nn.Linear(ffn_dim,d_model),nn.Dropout(dropout))
        self.resid_drop=nn.Dropout(dropout)
        self.adapter_attn=HoulsbyAdapter(d_model,ADAPTER_DIM)
        self.adapter_ff=HoulsbyAdapter(d_model,ADAPTER_DIM)
    def forward(self,x,attn_mask=None,past_kv=None,use_cache=False):
        a,new_kv=self.attn(self.ln1(x),attn_mask=attn_mask,past_kv=past_kv,use_cache=use_cache)
        x=self.adapter_attn(x+self.resid_drop(a))
        x=self.adapter_ff(x+self.ff(self.ln2(x)))
        return x,new_kv

class PrefixEncoderLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.vocab_size=len(tokenizer); self.d_model=D_MODEL; self.max_positions=MAX_POSITIONS
        self.token_emb=nn.Embedding(self.vocab_size,D_MODEL)
        self.pos=SinusoidalPositionalEncoding(D_MODEL,MAX_POSITIONS)
        self.drop=nn.Dropout(DROPOUT)
        self.blocks=nn.ModuleList([EncoderBlock(D_MODEL,N_HEADS,FFN_DIM,DROPOUT) for _ in range(N_LAYERS)])
        self.final_ln=nn.LayerNorm(D_MODEL)
    @staticmethod
    def prefix_mask(T,p,device):
        q=torch.arange(T,device=device)[:,None]; k=torch.arange(T,device=device)[None,:]
        return ((((q<p)&(k<p))|((q>=p)&(k<=q)))[None,None,:,:])
    def forward_hidden(self,input_ids,prefix_len=None,past_key_values=None,use_cache=False,
                       position_ids=None,key_valid=None):
        B,T=input_ids.shape; past_len=0 if past_key_values is None else past_key_values[0][0].size(2)
        if position_ids is None: position_ids=torch.arange(past_len,past_len+T,device=input_ids.device)[None,:].expand(B,T)
        x=self.token_emb(input_ids)+self.pos(position_ids).to(self.token_emb.weight.dtype); x=self.drop(x)
        if key_valid is not None: attn_mask=key_valid[:,None,None,:]
        elif past_key_values is None and prefix_len is not None and int(prefix_len)<T:
            attn_mask=self.prefix_mask(T,prefix_len,input_ids.device)
        else: attn_mask=None
        cache=[] if use_cache else None
        for i,block in enumerate(self.blocks):
            past=None if past_key_values is None else past_key_values[i]
            x,kv=block(x,attn_mask,past,use_cache)
            if use_cache: cache.append(kv)
        return self.final_ln(x),cache
    def project(self,h): return F.linear(h,self.token_emb.weight)

model=PrefixEncoderLM().to(DEVICE)
ckpt=torch.load(BASE_CKPT,map_location="cpu",weights_only=False)
state=ckpt.get("model",ckpt.get("state_dict",ckpt))
missing,unexpected=model.load_state_dict(state,strict=False)
print("Loaded Phase 1 weights. Missing adapter keys expected:",len(missing),"unexpected:",len(unexpected))
bad_missing=[k for k in missing if "adapter_" not in k]
assert not bad_missing, f"Unexpected missing base weights: {bad_missing[:10]}"
# Freeze all pretrained weights; train only Houlsby adapters.
for p in model.parameters(): p.requires_grad=False
for name,p in model.named_parameters():
    if "adapter_attn" in name or "adapter_ff" in name: p.requires_grad=True
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
total=sum(p.numel() for p in model.parameters())
print(f"Trainable adapter params: {trainable/1e6:.2f}M / {total/1e6:.1f}M ({100*trainable/total:.2f}%)")
assert trainable>0

Loaded Phase 1 weights. Missing adapter keys expected: 112 unexpected: 0
Trainable adapter params: 2.78M / 120.9M (2.30%)


In [8]:
# ---- Build compact supervised examples. Source is head+tail; target is reference summary.
MAX_SOURCE_TOKENS=512
MAX_TARGET_TOKENS=256
MAX_SEQ_LEN=MAX_SOURCE_TOKENS+MAX_TARGET_TOKENS
def encode_source(text, budget=MAX_SOURCE_TOKENS):
    ids=tokenizer.encode(text)
    if len(ids)<=budget: return ids
    h=budget*2//3; t=budget-h
    return ids[:h]+ids[-t:]
def make_pair(r):
    src=encode_source(r["document"])
    tgt=tokenizer.encode(r["summary"])[:MAX_TARGET_TOKENS]
    if not src or not tgt: return None
    if len(src)+len(tgt)>MAX_SEQ_LEN: tgt=tgt[:MAX_SEQ_LEN-len(src)]
    return src,tgt

train_pairs=[x for x in (make_pair(r) for r in train_all) if x]
val_pairs=[x for x in (make_pair(r) for r in final_val_rows) if x]
print("Encoded train/val:",len(train_pairs),len(val_pairs),
      "| avg source tokens:",int(np.mean([len(x[0]) for x in train_pairs])),
      "| avg target tokens:",int(np.mean([len(x[1]) for x in train_pairs])))
class SummDataset(Dataset):
    def __init__(self,pairs): self.pairs=pairs
    def __len__(self): return len(self.pairs)
    def __getitem__(self,i): return self.pairs[i]
def collate(batch):
    B=len(batch); L=max(len(s)+len(t) for s,t in batch)
    ids=torch.full((B,L),tokenizer.pad_token_id,dtype=torch.long)
    labels=torch.full((B,L-1),-100,dtype=torch.long)
    prefix=[]
    for i,(s,t) in enumerate(batch):
        seq=s+t+[tokenizer.eos_token_id]
        ids[i,:len(seq)]=torch.tensor(seq)
        # At prediction position len(source)-1, predict first target token.
        y_start=len(s)-1
        y_end=len(s)+len(t)
        labels[i,y_start:y_end]=torch.tensor(t+[tokenizer.eos_token_id])
        prefix.append(len(s))
    return ids[:,:-1],ids[:,1:],labels,torch.tensor(prefix,dtype=torch.long)

MICRO_BATCH=1
GRAD_ACCUM=8
EPOCHS=1   # deliberately one pass to respect the 90-minute budget
LR=2e-4
loader=DataLoader(SummDataset(train_pairs),batch_size=MICRO_BATCH,shuffle=True,
                  collate_fn=collate,num_workers=0,pin_memory=True)
val_loader=DataLoader(SummDataset(val_pairs),batch_size=1,shuffle=False,collate_fn=collate)
optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                            lr=LR,weight_decay=0.01)
steps_per_epoch=math.ceil(len(loader)/GRAD_ACCUM)
total_steps=steps_per_epoch*EPOCHS
warmup=max(1,int(total_steps*0.05))
scheduler=torch.optim.lr_scheduler.LambdaLR(
    optimizer,lambda s: min(1.0,(s+1)/warmup) if s<warmup else
    max(0.1,0.5*(1+math.cos(math.pi*(s-warmup)/max(1,total_steps-warmup)))))
scaler=torch.amp.GradScaler("cuda")
print("Optimizer steps:",total_steps,"| estimated train examples:",len(train_pairs))

Encoded train/val: 900 16 | avg source tokens: 511 | avg target tokens: 255
Optimizer steps: 113 | estimated train examples: 900


In [10]:

# Fix collate padding: reserve one extra position for EOS
def collate(batch):
    B = len(batch)
    L = max(len(s) + len(t) + 1 for s, t in batch)

    ids = torch.full(
        (B, L), tokenizer.pad_token_id, dtype=torch.long
    )
    labels = torch.full((B, L-1), -100, dtype=torch.long)
    prefix = []

    for i, (s, t) in enumerate(batch):
        seq = s + t + [tokenizer.eos_token_id]
        ids[i, :len(seq)] = torch.tensor(seq)

        y_start = len(s) - 1
        y_end = len(s) + len(t)
        labels[i, y_start:y_end] = torch.tensor(
            t + [tokenizer.eos_token_id]
        )
        prefix.append(len(s))

    return (
        ids[:, :-1],
        ids[:, 1:],
        labels,
        torch.tensor(prefix, dtype=torch.long)
    )

# Recreate loaders using corrected collate
loader = DataLoader(
    SummDataset(train_pairs),
    batch_size=MICRO_BATCH,
    shuffle=True,
    collate_fn=collate,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    SummDataset(val_pairs),
    batch_size=1,
    shuffle=False,
    collate_fn=collate
)

# Verify one batch before training
xb, yb, lb, pb = next(iter(loader))
assert xb.shape == yb.shape == lb.shape
assert xb.shape[1] == MAX_SEQ_LEN
print("Collate fix passed:", xb.shape, "Prefix:", pb.shape)


Collate fix passed: torch.Size([1, 768]) Prefix: torch.Size([1])


In [11]:
# ---- Train adapters only. Validation/checkpoint every 50 optimizer steps.
def evaluate_loss(max_items=32):
    model.eval(); losses=[]
    with torch.no_grad():
        for j,b in enumerate(val_loader):
            if j>=max_items: break
            x,y,lab,prefix=[z.to(DEVICE) for z in b]
            with torch.autocast("cuda",dtype=torch.float16):
                h,_=model.forward_hidden(x,prefix_len=int(prefix[0]))
                logits=model.project(h).float()
                loss=F.cross_entropy(logits.reshape(-1,logits.size(-1)),
                                     lab.reshape(-1),ignore_index=-100)
            losses.append(float(loss.item()))
    model.train()
    return float(np.mean(losses)) if losses else float("inf")

best_val=float("inf"); global_step=0; t0=time.time()
optimizer.zero_grad(set_to_none=True)
for epoch in range(EPOCHS):
    running=0.0; seen=0
    pbar=tqdm(enumerate(loader),total=len(loader),desc=f"Epoch {epoch+1}/{EPOCHS}")
    for it,b in pbar:
        x,y,lab,prefix=[z.to(DEVICE,non_blocking=True) for z in b]
        with torch.autocast("cuda",dtype=torch.float16):
            h,_=model.forward_hidden(x,prefix_len=int(prefix[0]))
            logits=model.project(h).float()
            loss=F.cross_entropy(logits.reshape(-1,logits.size(-1)),
                                 lab.reshape(-1),ignore_index=-100)
            loss=loss/GRAD_ACCUM
        scaler.scale(loss).backward()
        running+=float(loss.item())*GRAD_ACCUM; seen+=1
        if (it+1)%GRAD_ACCUM==0 or (it+1)==len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],1.0)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step(); global_step+=1
            pbar.set_postfix(loss=f"{running/max(1,seen):.3f}",step=global_step,
                             lr=f"{scheduler.get_last_lr()[0]:.2g}")
            if global_step%25==0 or global_step==total_steps:
                v=evaluate_loss()
                print(f"step={global_step}/{total_steps} train_loss={running/max(1,seen):.4f} val_loss={v:.4f} elapsed_min={(time.time()-t0)/60:.1f}")
                torch.save({"model":model.state_dict(),"global_step":global_step,
                            "val_loss":v,"config":{"adapter_dim":ADAPTER_DIM}},ROOT/"latest.pt")
                if v<best_val:
                    best_val=v
                    torch.save({"model":model.state_dict(),"global_step":global_step,
                                "val_loss":v,"config":{"adapter_dim":ADAPTER_DIM}},ROOT/"best.pt")
    print(f"Epoch {epoch+1} complete | elapsed={(time.time()-t0)/60:.1f} min")
# Always save final adapter state.
torch.save({"model":model.state_dict(),"global_step":global_step,"val_loss":best_val,
            "config":{"adapter_dim":ADAPTER_DIM}},ROOT/"final.pt")
print("Best validation loss:",best_val,"| training minutes:",(time.time()-t0)/60)

Epoch 1/1:   0%|          | 0/900 [00:00<?, ?it/s]

step=25/113 train_loss=2.7031 val_loss=2.6651 elapsed_min=0.2
step=50/113 train_loss=2.5856 val_loss=2.6059 elapsed_min=0.4
step=75/113 train_loss=2.5186 val_loss=2.5833 elapsed_min=0.6
step=100/113 train_loss=2.4892 val_loss=2.5749 elapsed_min=0.8
step=113/113 train_loss=2.4787 val_loss=2.5733 elapsed_min=0.9
Epoch 1 complete | elapsed=0.9 min
Best validation loss: 2.5732756555080414 | training minutes: 0.8943846146265666


In [12]:
# ---- Restore best adapter checkpoint (if it exists) and prepare generation.
best_path=ROOT/"best.pt"
if best_path.exists():
    obj=torch.load(best_path,map_location="cpu",weights_only=False)
    model.load_state_dict(obj["model"])
    print("Loaded best adapter checkpoint:",obj.get("global_step"),"val_loss:",obj.get("val_loss"))
model.eval()

@torch.inference_mode()
def generate_one(document, min_frac=0.0, no_repeat_ngram=0, repetition_penalty=1.0,
                 max_new_tokens=256, target_min_words=0):
    src=encode_source(document, MAX_SOURCE_TOKENS)
    x=torch.tensor([src],dtype=torch.long,device=DEVICE)
    with torch.autocast("cuda",dtype=torch.float16):
        h,cache=model.forward_hidden(x,prefix_len=len(src),use_cache=True)
    last=h[:,-1,:]
    generated=[]
    min_words=max(target_min_words,int(min_frac*96))
    for step in range(max_new_tokens):
        logits=model.project(last).float()
        logits[:,[tokenizer.pad_token_id,tokenizer.unk_token_id,tokenizer.bos_token_id]]=-float("inf")
        if repetition_penalty!=1.0 and generated:
            idx=torch.tensor([list(set(generated))],device=DEVICE)
            vals=logits.gather(1,idx)
            logits.scatter_(1,idx,torch.where(vals<0,vals*repetition_penalty,vals/repetition_penalty))
        if no_repeat_ngram>0 and len(generated)>=no_repeat_ngram-1:
            prefix=tuple(generated[-(no_repeat_ngram-1):]) if no_repeat_ngram>1 else tuple()
            banned=set()
            for j in range(len(generated)-no_repeat_ngram+1):
                if tuple(generated[j:j+no_repeat_ngram-1])==prefix:
                    banned.add(generated[j+no_repeat_ngram-1])
            if banned: logits[:,list(banned)]=-float("inf")
        nxt=int(logits.argmax(-1).item())
        if nxt==tokenizer.eos_token_id and len(tokenizer.decode(generated).split())>=min_words:
            break
        generated.append(nxt)
        if (step+1)%8==0:
            words=tokenizer.decode(generated).split()
            if len(words)>=240 and step>32:
                break
        nt=torch.tensor([[nxt]],device=DEVICE)
        with torch.autocast("cuda",dtype=torch.float16):
            h,cache=model.forward_hidden(nt,past_key_values=cache,use_cache=True)
        last=h[:,-1,:]
    return tokenizer.decode(generated).strip()

# Three compact configurations; tune on the 16 decode-tuning documents to limit runtime.
DECODE_CONFIGS=[
    {"name":"greedy","min_frac":0.0,"ngram":0,"rep":1.0},
    {"name":"ng8+rep1.05+min","min_frac":0.60,"ngram":8,"rep":1.05},
]
TUNE_N=min(6,len(decode_rows))
tune_rows=decode_rows[:TUNE_N]
print("Decode tuning docs:",TUNE_N,"| configs:",[x["name"] for x in DECODE_CONFIGS])

Loaded best adapter checkpoint: 113 val_loss: 2.5732756555080414
Decode tuning docs: 6 | configs: ['greedy', 'ng8+rep1.05+min']


In [13]:
# ---- Run the small decoding sweep and select by BERTScore F1.
import importlib, subprocess, sys
def _ensure(mod, pip):
    try: importlib.import_module(mod)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pip])
_ensure("rouge_score","rouge-score"); _ensure("bert_score","bert-score")
from rouge_score import rouge_scorer
_rs=rouge_scorer.RougeScorer(["rouge1","rougeL"],use_stemmer=True)
def score_all(preds, refs):
    from bert_score import score as bertscore
    safe=[p if p.strip() else "." for p in preds]
    _,_,f=bertscore(safe,refs,lang="en",batch_size=8,device=str(DEVICE),verbose=False)
    sc=[_rs.score(r,p) for p,r in zip(preds,refs)]
    m={"rouge1_f1":float(np.mean([s["rouge1"].fmeasure for s in sc])),
       "rougeL_f1":float(np.mean([s["rougeL"].fmeasure for s in sc])),
       "bertscore_f1":float(f.mean().item())}
    m["composite"]=(m["rouge1_f1"]+m["rougeL_f1"]+m["bertscore_f1"])/3
    return m
def score_predictions(preds, refs): return score_all(preds,refs)["bertscore_f1"]

tune_results=[]
for cfg in DECODE_CONFIGS:
    t0=time.time()
    preds=[generate_one(r["document"],cfg["min_frac"],cfg["ngram"],cfg["rep"])
           for r in tqdm(tune_rows,desc=cfg["name"])]
    refs=[r["summary"] for r in tune_rows]
    ms=score_all(preds,refs)
    result={**cfg,**ms,"avg_words":float(np.mean([len(p.split()) for p in preds])),
            "seconds":time.time()-t0}
    tune_results.append(result)
    print(result)
best_cfg=max(tune_results,key=lambda x:x["bertscore_f1"])
print("\nSelected decoding by BERTScore F1:",best_cfg)
json.dump({"results":tune_results,"selected":best_cfg},open(ROOT/"decode_tuning.json","w"),indent=2)

greedy:   0%|          | 0/6 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (5 outputs left out)

ng8+rep1.05+min:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'name': 'ng8+rep1.05+min', 'min_frac': 0.6, 'ngram': 8, 'rep': 1.05, 'rouge1_f1': 0.1566670551721356, 'rougeL_f1': 0.11043269444279989, 'bertscore_f1': 0.7928324341773987, 'composite': 0.35331072793077806, 'avg_words': 127.5, 'seconds': 21.974685668945312}

Selected decoding by BERTScore F1: {'name': 'ng8+rep1.05+min', 'min_frac': 0.6, 'ngram': 8, 'rep': 1.05, 'rouge1_f1': 0.1566670551721356, 'rougeL_f1': 0.11043269444279989, 'bertscore_f1': 0.7928324341773987, 'composite': 0.35331072793077806, 'avg_words': 127.5, 'seconds': 21.974685668945312}


In [14]:
# ---- Final held-out validation: separate from the 6 tuning documents.
t0=time.time()
val_preds=[]
for r in tqdm(final_val_rows,desc="Final validation"):
    val_preds.append(generate_one(r["document"],best_cfg["min_frac"],best_cfg["ngram"],best_cfg["rep"]))
val_refs=[r["summary"] for r in final_val_rows]
val_metrics=score_all(val_preds,val_refs); val_bert=val_metrics["bertscore_f1"]
print("FINAL VALIDATION:",val_metrics)
print({
    "documents":len(final_val_rows),
    "bertscore_f1":val_bert,
    "avg_pred_words":float(np.mean([len(x.split()) for x in val_preds])),
    "avg_ref_words":float(np.mean([len(x.split()) for x in val_refs])),
    "generation_min":(time.time()-t0)/60
})
json.dump([{"id":r["id"],"summary":p} for r,p in zip(final_val_rows,val_preds)],
          open(ROOT/"validation_predictions.json","w"),ensure_ascii=False,indent=2)

Final validation:   0%|          | 0/16 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


FINAL VALIDATION: {'rouge1_f1': 0.17780190388761702, 'rougeL_f1': 0.12761432845873966, 'bertscore_f1': 0.7861436605453491, 'composite': 0.3638532976305686}
{'documents': 16, 'bertscore_f1': 0.7861436605453491, 'avg_pred_words': 124.4375, 'avg_ref_words': 845.0625, 'generation_min': 0.8943623185157776}


In [15]:
# ---- Official test generation + submission ZIP.
t0=time.time()
FALLBACK_SUMMARY="The judgment could not be summarized from the supplied context."
partial=ROOT/"submission_partial.jsonl"; done={}
if partial.exists():
    for line in open(partial,encoding="utf-8"):
        if line.strip():
            d=json.loads(line); done[d["id"]]=d["summary"]
    print("Resuming with",len(done),"cached test predictions")
with open(partial,"a",encoding="utf-8") as pf:
    for r in tqdm(test_rows,desc="Official test"):
        if r["id"] in done: continue
        summary=generate_one(r["document"],best_cfg["min_frac"],best_cfg["ngram"],best_cfg["rep"]).strip() or FALLBACK_SUMMARY
        done[r["id"]]=summary
        pf.write(json.dumps({"id":r["id"],"summary":summary},ensure_ascii=False)+"\n"); pf.flush()
test_predictions=[{"id":r["id"],"summary":done[r["id"]]} for r in test_rows]

assert len(test_predictions)==100
assert len({x["id"] for x in test_predictions})==100
assert all(x["summary"].strip() for x in test_predictions)
pred_path=ROOT/"prediction.json"
with open(pred_path,"w",encoding="utf-8") as f:
    json.dump(test_predictions,f,ensure_ascii=False,indent=2)
zip_path=ROOT/"phase6_exp4_model1_houlsby_submission.zip"
with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_path,arcname="prediction.json")
with zipfile.ZipFile(zip_path) as zf:
    assert zf.namelist()==["prediction.json"]
print("Predictions:",len(test_predictions))
print("Average words:",np.mean([len(x["summary"].split()) for x in test_predictions]))
print("Generation minutes:",(time.time()-t0)/60)
print("Submission ZIP:",zip_path,"| MB:",zip_path.stat().st_size/1024**2)

Official test:   0%|          | 0/100 [00:00<?, ?it/s]

Predictions: 100
Average words: 126.82
Generation minutes: 5.300129906336466
Submission ZIP: /kaggle/working/phase6_exp4_model1_houlsby/phase6_exp4_model1_houlsby_submission.zip | MB: 0.0119171142578125


## Runtime controls and cautions

- Lean configuration: **1 epoch**, at most **900 training documents**, 512 source tokens and 256 target tokens.
- Decoding tuning uses only 6 documents and 2 configurations.
- Final validation uses 16 separate documents; official test generation still covers all 100 test records.
- Generation is one document at a time to keep the custom prefix-cache path simple and reliable.
- If actual speed is slower than expected, do not submit partial predictions. All 100 test records are mandatory.
- Verify that Cell 2 resolves the matching Model 1 `enc_full_best` checkpoint and its tokenizer. Do not attach Model 2 or Model 4 weights.
- This is a deliberately modest experiment; no exhaustive search is performed.


In [16]:
# ---- Save experiment metadata + prediction sanity checks.
lens=np.array([len(x["summary"].split()) for x in test_predictions])
print(f"Test words mean={lens.mean():.0f} min={lens.min()} max={lens.max()} | fallbacks={sum(x['summary']==FALLBACK_SUMMARY for x in test_predictions)}")
json.dump({"experiment":"Phase6-Exp4-Model1-Houlsby","adapter_dim":ADAPTER_DIM,"epochs":EPOCHS,"max_source_tokens":MAX_SOURCE_TOKENS,"max_target_tokens":MAX_TARGET_TOKENS,"lr":LR,
           "grad_accum":GRAD_ACCUM,"max_source_tokens":MAX_SOURCE_TOKENS,"max_target_tokens":MAX_TARGET_TOKENS,
           "train_examples":len(train_pairs),"best_val_loss":best_val,"selected_decoding":best_cfg,
           "decode_tuning":tune_results,"validation_metrics":val_metrics,"seed":SEED},
          open(ROOT/"experiment_metadata.json","w"),indent=2)


Test words mean=127 min=84 max=128 | fallbacks=0
